In [1]:
import json
import torch
from sentence_transformers import SentenceTransformer, util

/home/gusevsaint/Workspace/study/practice/mental-helper/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "intfloat/e5-small-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)

In [ ]:
with open("../data/qa_dataset.json", "r", encoding="utf-8") as f:
    qa_data = json.load(f)

questions = [f"passage: {item['question']}" for item in qa_data]

embeddings = model.encode(
    questions,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print(f"Индекс построен: {len(embeddings)} вопросов, размер={embeddings.shape[1]}")

Batches: 100%|██████████| 6/6 [00:01<00:00,  5.99it/s]

Индекс построен: 172 вопросов, размер=384


In [ ]:
def retrieve(query: str, top_k: int = 3):
    q_emb = model.encode(
        f"query: {query}", convert_to_tensor=True, normalize_embeddings=True
    )
    hits = util.semantic_search(q_emb, embeddings, top_k=top_k)[0]
    results = []
    for h in hits:
        item = qa_data[h["corpus_id"]]
        results.append(
            {
                "score": float(h["score"]),
                "id": item.get("id", h["corpus_id"]),
                "question": item["question"],
                "answer": item["answer"],
            }
        )
    return results


for i, r in enumerate(retrieve("How to cope with a stress?", top_k=3), 1):
    print(f"\n({i}) score={r['score']:.3f}")
    print(f"Q: {r['question']}")
    print(f"A: {r['answer'][:200]}")


(1) score=0.928
Q: How to manage stress?
A: Here are some personalized suggestions to help you cope with stress:

1. Identify Triggers: Take some time to identify what triggers your stress. This awareness can help you anticipate stressful situa

(2) score=0.910
Q: Are There Coping Factors To Help Deal Effectively With Stress?
A: Absolutely, coping factors can be immensely helpful in dealing effectively with stress. Here are some coping strategies that can be beneficial:

1. Mindfulness and Relaxation Techniques: Practicing mi

(3) score=0.876
Q: How to cope up with social isolation?
A: No matter how old you are, it’s important to recognize when you struggle with social isolation. Noticing is the first step to developing healthy coping mechanisms. To cope with social isolation, try t


In [ ]:
torch.save({"embeddings": embeddings, "model_name": model_name}, "../data/e5_index.pt")